# NN demo: MscaleCNN reflectivity prediction

Noiseless 10-epoch neural-network demo using `seis.mat` as input and `rbp_180_200.mat` as the high-resolution target. For a compact public demo, the model is trained on one fixed profile trace across 30 intervals, so the output figure has the same fan-shaped layout as the paper figures.

In [ ]:
EPOCHS = 10
PROFILE_TRACE = 100
TRACE_START = 0
TRACE_COUNT = 30
TIME_START = 200
TIME_STOP = 800
DT = 0.001
T0 = TIME_START * DT


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from demo_utils import (
    DATA_DIR,
    FIGURE_DIR,
    RESULT_DIR,
    extract_positions,
    load_mat_array,
    pick_interval_section,
    plot_position_points,
    save_mat,
    sparse_spike_deconvolution,
    to_interval_trace_time,
    train_position_demo,
    train_reflectivity_demo,
    wigb,
)

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 0.9,
    "savefig.dpi": 300,
})
FIGURE_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)


## Load noiseless data

In [ ]:
rbp = to_interval_trace_time(load_mat_array(DATA_DIR / "rbp_180_200.mat", "rbp_180_200"))
seis = to_interval_trace_time(load_mat_array(DATA_DIR / "seis.mat", "seis"))

seis_sec = pick_interval_section(seis, PROFILE_TRACE, TRACE_START, TRACE_COUNT, TIME_START, TIME_STOP)
rbp_sec = pick_interval_section(rbp, PROFILE_TRACE, TRACE_START, TRACE_COUNT, TIME_START, TIME_STOP)
print("seis", seis.shape, "rbp", rbp.shape)

## Run 10-epoch NN demo on one profile

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from core.Mscale_cnnnet import MscaleCNN
from demo_utils import normalize_traces

torch.manual_seed(10)
device = "cuda:0" if torch.cuda.is_available() else "cpu"

x_all = seis[:, PROFILE_TRACE, :].astype(np.float32)
y_all = rbp[:, PROFILE_TRACE, :].astype(np.float32)
x_norm, _, _ = normalize_traces(x_all)
y_norm, y_mean, y_std = normalize_traces(y_all)

loader = DataLoader(
    TensorDataset(torch.from_numpy(x_norm), torch.from_numpy(y_norm)),
    batch_size=1,
    shuffle=True,
)
model = MscaleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.L1Loss()
losses = []

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        pred, dec1, dec2 = model(x_batch)
        target2 = F.interpolate(y_batch.view(-1, 1, y_batch.shape[-1]), scale_factor=2, mode="linear")
        loss = criterion(pred, y_batch) + 0.1 * criterion(dec1, y_batch)
        loss = loss + 0.1 * criterion(dec2, target2.view(dec2.shape))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss.detach().cpu())
    losses.append(epoch_loss / len(loader))

predictions = []
model.eval()
with torch.no_grad():
    for trace in torch.from_numpy(x_norm).to(device):
        pred, _, _ = model(trace.unsqueeze(0))
        predictions.append(pred.squeeze(0).cpu().numpy())
profile_prediction = np.stack(predictions, axis=0) * y_std + y_mean
save_mat(RESULT_DIR / "quick_nn_profile_noiseless.mat", prediction=profile_prediction.astype(np.float32), loss=np.asarray(losses, dtype=np.float32))
print("profile_prediction", profile_prediction.shape)
print("Final loss:", losses[-1])

## Paper-style NN figure

In [ ]:
pred_sec = profile_prediction[:, TIME_START:TIME_STOP].T

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
wigb(seis_sec, dt=DT, t0=T0, scale=0.75, ax=axes[0], linewidth=0.6, panel_label="(a)")
wigb(rbp_sec, dt=DT, t0=T0, scale=0.75, ax=axes[1], linewidth=0.6, panel_label="(b)")
wigb(pred_sec, dt=DT, t0=T0, scale=0.75, ax=axes[2], linewidth=0.6, panel_label="(c)")
for ax in axes[1:]:
    ax.set_ylabel("")
fig.savefig(FIGURE_DIR / "nn_demo.png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, EPOCHS + 1), losses, "o-", linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("L1 loss")
plt.tick_params(direction="in", top=True, right=True)
plt.savefig(FIGURE_DIR / "nn_loss.png", bbox_inches="tight")
plt.show()